Pré-processamento

In [19]:
!pip install mediapipe opencv-python pandas numpy tqdm

Defaulting to user installation because normal site-packages is not writeable


In [20]:
import os
import json
from collections import Counter
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm

In [21]:
# =========================
# CONFIGURAÇÃO
# =========================

# Ajusta estes caminhos ao teu dataset real
TRAIN_DIR = "ASL_Alphabet_Dataset/asl_alphabet_train"
TEST_DIR = "ASL_Alphabet_Dataset/asl_alphabet_test_2"

OUTPUT_TRAIN_CSV = "outputs/train_landmarks.csv"
OUTPUT_TEST_CSV = "outputs/test_landmarks.csv"
OUTPUT_TRAIN_STATS = "outputs/train_stats.csv"
OUTPUT_TEST_STATS = "outputs/test_stats.csv"
OUTPUT_EXTRACTION_SUMMARY = "outputs/extraction_summary.json"

CLASSES = [chr(c) for c in range(ord('A'), ord('Z') + 1)] # + ["space", "nothing", "del"]

# Tamanhos a testar por ordem
SIZES_PRIMARY = [256, 384, 512]
SIZES_SECONDARY = [384, 512]

# Coloca None para processar tudo
MAX_IMAGES_PER_CLASS = 300 # None 

# =========================
# MEDIAPIPE HANDS
# =========================
mp_hands = mp.solutions.hands

hands_primary = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.35
)

hands_relaxed = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.20
)

I0000 00:00:1773338717.797013   27350 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1773338717.798452   45799 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)
I0000 00:00:1773338717.802390   27350 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1773338717.803385   45818 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)


In [22]:
# =========================
# FUNÇÕES AUXILIARES
# =========================

def get_border_median_color(image, border=10):
    h, w = image.shape[:2]
    border = max(1, min(border, h, w))

    top = image[:border, :, :]
    bottom = image[h-border:h, :, :]
    left = image[:, :border, :]
    right = image[:, w-border:w, :]

    pixels = np.concatenate([
        top.reshape(-1, 3),
        bottom.reshape(-1, 3),
        left.reshape(-1, 3),
        right.reshape(-1, 3)
    ], axis=0)

    median_color = np.median(pixels, axis=0)
    return tuple(int(x) for x in median_color)


def resize_and_pad(image, target_size=256):
    h, w = image.shape[:2]

    scale = target_size / max(h, w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(image, (new_w, new_h), interpolation=interp)

    delta_w = target_size - new_w
    delta_h = target_size - new_h

    top = delta_h // 2
    bottom = delta_h - top
    left = delta_w // 2
    right = delta_w - left

    pad_color = get_border_median_color(image)

    padded = cv2.copyMakeBorder(
        resized,
        top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT,
        value=pad_color
    )
    return padded


def apply_clahe_bgr(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l2 = clahe.apply(l)
    lab2 = cv2.merge((l2, a, b))
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)


def sharpen_image(image):
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]], dtype=np.float32)
    return cv2.filter2D(image, -1, kernel)


def estimate_hand_bbox_from_landmarks(landmarks, image_shape, margin=0.20):
    h, w = image_shape[:2]
    xs = [lm.x for lm in landmarks]
    ys = [lm.y for lm in landmarks]

    min_x = max(0.0, min(xs) - margin)
    max_x = min(1.0, max(xs) + margin)
    min_y = max(0.0, min(ys) - margin)
    max_y = min(1.0, max(ys) + margin)

    x1 = int(min_x * w)
    y1 = int(min_y * h)
    x2 = int(max_x * w)
    y2 = int(max_y * h)

    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2

"""
def normalize_landmarks(lms):
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z

    # Escala robusta: média das distâncias do pulso às bases dos dedos
    base_ids = [5, 9, 13, 17]
    dists = []
    for idx in base_ids:
        dx = lms[idx].x - wx
        dy = lms[idx].y - wy
        dz = lms[idx].z - wz
        dists.append((dx**2 + dy**2 + dz**2) ** 0.5)

    scale = float(np.mean(dists))
    if scale < 1e-6:
        return None

    features = []
    for lm in lms:
        features.extend([
            (lm.x - wx) / scale,
            (lm.y - wy) / scale,
            (lm.z - wz) / scale
        ])
    return features
"""

"""
def normalize_landmarks(lms):
    # 1. Referência ao pulso (ponto 0)
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z

    # 2. Cálculo da escala (para ser invariante à distância da câmara)
    base_ids = [5, 9, 13, 17]
    dists = []
    for idx in base_ids:
        dx = lms[idx].x - wx
        dy = lms[idx].y - wy
        dz = lms[idx].z - wz
        dists.append((dx**2 + dy**2 + dz**2) ** 0.5)

    scale = float(np.mean(dists))
    
    # Prevenção de erro de divisão por zero
    if scale < 1e-6:
        return None

    # 3. Normalização: Relativa ao pulso E ajustada pela escala
    features = []
    for lm in lms:
        features.extend([
            (lm.x - wx) / scale,
            (lm.y - wy) / scale,
            (lm.z - wz) / scale
        ])
    return features
"""


def normalize_landmarks(lms):
    # 1. Coordenadas Relativas ao Pulso (Ponto 0)
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z

    # 2. Cálculo da Escala Robusta (idêntico ao que já tinhas)
    base_ids = [5, 9, 13, 17]
    dists_base = []
    for idx in base_ids:
        dx = lms[idx].x - wx
        dy = lms[idx].y - wy
        dz = lms[idx].z - wz
        dists_base.append((dx**2 + dy**2 + dz**2) ** 0.5)

    scale = float(np.mean(dists_base))
    if scale < 1e-6: return None

    features = []
    
    # 3. Atributos de Coordenadas (63 valores)
    for lm in lms:
        features.extend([
            (lm.x - wx) / scale,
            (lm.y - wy) / scale,
            (lm.z - wz) / scale
        ])
    
    # 4. NOVOS ATRIBUTOS: Distâncias Euclidianas ao Pulso (21 valores)
    # Isto ajuda imenso a distinguir M, N, S porque foca na "compactação" da mão
    for lm in lms:
        d = ((lm.x - wx)**2 + (lm.y - wy)**2 + (lm.z - wz)**2)**0.5
        features.append(d / scale)

    return features # Agora retorna 84 valores em vez de 63



def run_hands_detector(image_bgr, detector):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return detector.process(image_rgb)


def detect_landmarks_with_retries(image_bgr):
    # 1) Original
    for size in SIZES_PRIMARY:
        candidate = resize_and_pad(image_bgr, target_size=size)
        results = run_hands_detector(candidate, hands_primary)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"original_{size}_primary", candidate

    # 2) CLAHE
    image_clahe = apply_clahe_bgr(image_bgr)
    for size in SIZES_PRIMARY:
        candidate = resize_and_pad(image_clahe, target_size=size)
        results = run_hands_detector(candidate, hands_primary)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"clahe_{size}_primary", candidate

    # 3) Sharpen
    image_sharp = sharpen_image(image_bgr)
    for size in SIZES_SECONDARY:
        candidate = resize_and_pad(image_sharp, target_size=size)
        results = run_hands_detector(candidate, hands_primary)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"sharp_{size}_primary", candidate

    # 4) Detector mais permissivo na original
    for size in SIZES_PRIMARY:
        candidate = resize_and_pad(image_bgr, target_size=size)
        results = run_hands_detector(candidate, hands_relaxed)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"original_{size}_relaxed", candidate

    # 5) Detector mais permissivo + CLAHE
    for size in SIZES_PRIMARY:
        candidate = resize_and_pad(image_clahe, target_size=size)
        results = run_hands_detector(candidate, hands_relaxed)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"clahe_{size}_relaxed", candidate

    return None, None, None


def crop_and_redetect(image_bgr):
    landmarks, method, processed = detect_landmarks_with_retries(image_bgr)
    if landmarks is None:
        return None, None

    bbox = estimate_hand_bbox_from_landmarks(landmarks, processed.shape, margin=0.20)
    if bbox is None:
        return landmarks, method

    x1, y1, x2, y2 = bbox
    crop = processed[y1:y2, x1:x2]
    if crop.size == 0:
        return landmarks, method

    crop = resize_and_pad(crop, target_size=384)
    results = run_hands_detector(crop, hands_primary)
    if results.multi_hand_landmarks:
        return results.multi_hand_landmarks[0].landmark, method + "_crop_refined"

    return landmarks, method


def extract_hand_landmarks(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None, "read_error"

    landmarks, method = crop_and_redetect(image)
    if landmarks is None:
        return None, "no_hand_detected"

    features = normalize_landmarks(landmarks)
    if features is None:
        return None, "invalid_scale"

    return features, method


def build_columns():
    cols = []
    # 1. Nomes para as 63 coordenadas (x0, y0, z0, ..., z20)
    for i in range(21):
        cols.extend([f"x{i}", f"y{i}", f"z{i}"])
    
    # 2. Nomes para as 21 novas distâncias (dist_0, ..., dist_20)
    for i in range(21):
        cols.append(f"dist_{i}")
        
    # 3. A etiqueta final
    cols.append("label")
    return cols


def collect_test_files(test_dir):
    rows = []

    flat_files = sorted(
        f for f in os.listdir(test_dir)
        if os.path.isfile(os.path.join(test_dir, f)) and f.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    for fname in flat_files:
        label = os.path.splitext(fname)[0]
        if label.endswith("_test"):
            label = label[:-5]
        rows.append((os.path.join(test_dir, fname), label))

    subdirs = sorted(
        d for d in os.listdir(test_dir)
        if os.path.isdir(os.path.join(test_dir, d))
    )
    for label in subdirs:
        class_dir = os.path.join(test_dir, label)
        image_files = sorted(
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )
        for fname in image_files:
            rows.append((os.path.join(class_dir, fname), label))

    return rows


def process_train_dataset(train_dir):
    rows = []
    failed = []
    stats = {}
    extraction_methods = Counter()

    for label in CLASSES:
        class_dir = os.path.join(train_dir, label)

        if not os.path.isdir(class_dir):
            print(f"[AVISO] Pasta não encontrada: {class_dir}")
            continue

        image_files = sorted(
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )

        if MAX_IMAGES_PER_CLASS is not None:
            image_files = image_files[:MAX_IMAGES_PER_CLASS]

        total = len(image_files)
        ok = 0
        bad = 0

        for img_name in tqdm(image_files, desc=f"Classe {label}"):
            img_path = os.path.join(class_dir, img_name)
            features, method = extract_hand_landmarks(img_path)

            if features is None:
                failed.append({"path": img_path, "reason": method})
                bad += 1
                continue

            rows.append(features + [label])
            extraction_methods[method] += 1
            ok += 1

        stats[label] = {
            "total": total,
            "ok": ok,
            "failed": bad,
            "success_rate": round(100 * ok / total, 2) if total > 0 else 0.0
        }

    return rows, failed, stats, extraction_methods


def process_test_dataset(test_dir):
    rows = []
    failed = []
    extraction_methods = Counter()
    per_label = Counter()

    if not os.path.isdir(test_dir):
        print(f"[AVISO] Pasta de teste não encontrada: {test_dir}")
        return rows, failed, {}, extraction_methods

    test_files = collect_test_files(test_dir)
    print(f"[INFO] Ficheiros de teste encontrados: {len(test_files)}")

    for img_path, label in tqdm(test_files, desc="Teste"):
        features, method = extract_hand_landmarks(img_path)

        if features is None:
            failed.append({"path": img_path, "reason": method})
            per_label[(label, "failed")] += 1
            continue

        rows.append(features + [label])
        extraction_methods[method] += 1
        per_label[(label, "ok")] += 1

    test_stats = {}
    labels = sorted({label for _, label in test_files})
    for label in labels:
        ok = per_label.get((label, "ok"), 0)
        bad = per_label.get((label, "failed"), 0)
        total = ok + bad
        test_stats[label] = {
            "total": total,
            "ok": ok,
            "failed": bad,
            "success_rate": round(100 * ok / total, 2) if total > 0 else 0.0
        }

    return rows, failed, test_stats, extraction_methods


def count_images_in_directory(root_dir):
    total = 0
    for current_root, _, files in os.walk(root_dir):
        total += sum(
            1 for f in files
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )
    return total


print("Imagens no treino:", count_images_in_directory(TRAIN_DIR) if os.path.isdir(TRAIN_DIR) else "diretório não encontrado")
print("Imagens no teste:", count_images_in_directory(TEST_DIR) if os.path.isdir(TEST_DIR) else "diretório não encontrado")

W0000 00:00:1773338717.823246   45803 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773338717.823520   45782 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773338717.839116   45812 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773338717.839989   45789 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Imagens no treino: 206137
Imagens no teste: 1815


In [23]:
def save_failed_log(path, failed_items):
    with open(path, "w", encoding="utf-8") as f:
        for item in failed_items:
            if isinstance(item, dict):
                f.write(f"{item['path']} | reason={item['reason']}\n")
            else:
                f.write(str(item) + "\n")


In [24]:
def main():
    os.makedirs("outputs", exist_ok=True)

    columns = build_columns()

    print(f"[INFO] TRAIN_DIR = {TRAIN_DIR}")
    print(f"[INFO] TEST_DIR  = {TEST_DIR}")

    # =========================
    # TREINO
    # =========================
    print("\nA processar treino...")
    train_rows, train_failed, train_stats, train_methods = process_train_dataset(TRAIN_DIR)
    df_train = pd.DataFrame(train_rows, columns=columns)
    df_train.to_csv(OUTPUT_TRAIN_CSV, index=False)

    # =========================
    # TESTE
    # =========================
    print("\nA processar teste...")
    test_rows, test_failed, test_stats, test_methods = process_test_dataset(TEST_DIR)
    df_test = pd.DataFrame(test_rows, columns=columns)
    df_test.to_csv(OUTPUT_TEST_CSV, index=False)

    # =========================
    # RESUMO GLOBAL
    # =========================
    print("\n===== RESUMO =====")
    print(f"Treino: {len(df_train)} amostras guardadas em {OUTPUT_TRAIN_CSV}")
    print(f"Teste:  {len(df_test)} amostras guardadas em {OUTPUT_TEST_CSV}")

    print(f"Falhas treino: {len(train_failed)}")
    print(f"Falhas teste:  {len(test_failed)}")

    # =========================
    # STATS TREINO
    # =========================
    print("\n===== SUCESSO POR CLASSE (TREINO) =====")
    for label, info in train_stats.items():
        print(
            f"{label}: {info['ok']}/{info['total']} "
            f"({info['success_rate']}%) - falhas: {info['failed']}"
        )

    # =========================
    # STATS TESTE
    # =========================
    if test_stats:
        print("\n===== SUCESSO POR CLASSE (TESTE) =====")
        for label, info in test_stats.items():
            print(
                f"{label}: {info['ok']}/{info['total']} "
                f"({info['success_rate']}%) - falhas: {info['failed']}"
            )

    # =========================
    # GUARDAR STATS EM CSV
    # =========================
    pd.DataFrame(train_stats).T.to_csv(OUTPUT_TRAIN_STATS)

    if test_stats:
        pd.DataFrame(test_stats).T.to_csv(OUTPUT_TEST_STATS)

    # =========================
    # GUARDAR LOGS DE FALHAS
    # =========================
    if train_failed:
        save_failed_log("outputs/train_failed.txt", train_failed)

    if test_failed:
        save_failed_log("outputs/test_failed.txt", test_failed)

    # =========================
    # MÉTODOS USADOS
    # =========================
    print("\n===== MÉTODOS DE EXTRAÇÃO USADOS (TREINO) =====")
    for method, count in train_methods.most_common():
        print(f"{method}: {count}")

    if test_methods:
        print("\n===== MÉTODOS DE EXTRAÇÃO USADOS (TESTE) =====")
        for method, count in test_methods.most_common():
            print(f"{method}: {count}")

    hands_primary.close()
    hands_relaxed.close()


if __name__ == "__main__":
    main()

[INFO] TRAIN_DIR = ASL_Alphabet_Dataset/asl_alphabet_train
[INFO] TEST_DIR  = ASL_Alphabet_Dataset/asl_alphabet_test_2

A processar treino...


Classe Z: 100%|██████████| 300/300 [00:14<00:00, 21.04it/s]



A processar teste...
[INFO] Ficheiros de teste encontrados: 1815


Teste: 100%|██████████| 1815/1815 [02:37<00:00, 11.53it/s]



===== RESUMO =====
Treino: 7672 amostras guardadas em outputs/train_landmarks.csv
Teste:  1495 amostras guardadas em outputs/test_landmarks.csv
Falhas treino: 128
Falhas teste:  320

===== SUCESSO POR CLASSE (TREINO) =====
A: 300/300 (100.0%) - falhas: 0
B: 300/300 (100.0%) - falhas: 0
C: 290/300 (96.67%) - falhas: 10
D: 296/300 (98.67%) - falhas: 4
E: 300/300 (100.0%) - falhas: 0
F: 300/300 (100.0%) - falhas: 0
G: 300/300 (100.0%) - falhas: 0
H: 300/300 (100.0%) - falhas: 0
I: 296/300 (98.67%) - falhas: 4
J: 271/300 (90.33%) - falhas: 29
K: 297/300 (99.0%) - falhas: 3
L: 299/300 (99.67%) - falhas: 1
M: 298/300 (99.33%) - falhas: 2
N: 287/300 (95.67%) - falhas: 13
O: 299/300 (99.67%) - falhas: 1
P: 277/300 (92.33%) - falhas: 23
Q: 281/300 (93.67%) - falhas: 19
R: 294/300 (98.0%) - falhas: 6
S: 297/300 (99.0%) - falhas: 3
T: 300/300 (100.0%) - falhas: 0
U: 296/300 (98.67%) - falhas: 4
V: 299/300 (99.67%) - falhas: 1
W: 300/300 (100.0%) - falhas: 0
X: 300/300 (100.0%) - falhas: 0
Y: 299